### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="wine_quality",
    dataset_year="2009",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C56S3T",
    download_description="""
Download from UCI and uzip data to a predefined folder.
mkdir -p local-data-warehouse/wine_quality/ && wget -P local-data-warehouse/wine_quality/ https://archive.ics.uci.edu/static/public/186/wine+quality.zip && unzip local-data-warehouse/wine_quality/wine+quality.zip -d local-data-warehouse/wine_quality/ && rm local-data-warehouse/wine_quality/wine+quality.zip
""",
    # References
    academic_reference_bibtex=r"""@article{cortez2009modeling,
  title={Modeling wine preferences by data mining from physicochemical properties},
  author={Cortez, Paulo and Cerdeira, Ant{\'o}nio and Almeida, Fernando and Matos, Telmo and Reis, Jos{\'e}},
  journal={Decision support systems},
  volume={47},
  number={4},
  pages={547--553},
  year={2009},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="cortez2009modeling",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We combine the original datasets for red and white wine into a single dataset with an additional column indicating the type of wine (red or white).
- We treat the task as a regression problem, following the original work and because the target is the median wine quality (of at least 3 evaluations by experts).
- Anomaly: the data has a high number of duplicates (18%).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="median_wine_quality",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd

df_red = pd.read_csv(dataset_mold.path / "winequality-red.csv", sep=";")
df_white = pd.read_csv(dataset_mold.path / "winequality-white.csv", sep=";")
df_red["wine_color"] = "red"
df_white["wine_color"] = "white"
df = pd.concat([df_white, df_red], ignore_index=True) 
df.columns = df.columns.str.replace(" ", "_")
df["wine_color"] = df["wine_color"].astype("category")

target_feature = "median_wine_quality"
df = df.rename(columns={"quality": target_feature})

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 6,497
Columns: 13
Use sampling: False (sample size: 6,497)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['density', 'residual_sugar', 'total_sulfur_dioxide', 'chlorides', 'volatile_acidity', 'free_sulfur_dioxide', 'sulphates', 'alcohol', 'pH', 'fixed_acidity']
Rows remaining as candidates after top-10 filter: 2,172 (of 6,497)

#### Duplicate Report
Total duplicate rows: 1177 (18.12% of dataset)
Duplicate rows ignoring target: 1177 (18.12% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,median_wine_quality,wine_color
0,7.0,0.25,0.45,2.3,0.045,40.0,118.0,0.99064,3.16,0.48,11.9,7,white
1,7.6,0.14,0.74,1.6,0.040,27.0,103.0,0.99160,3.07,0.40,10.8,7,white
2,6.2,0.15,0.27,11.0,0.035,46.0,116.0,0.99602,3.12,0.38,9.1,6,white
3,6.7,0.16,0.32,12.5,0.035,18.0,156.0,0.99666,2.88,0.36,9.0,6,white
4,6.8,0.27,0.22,17.8,0.034,16.0,116.0,0.99890,3.07,0.53,9.2,5,white


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,wine_color,category,0.0,0.0,2.0,"white, red"
1,fixed_acidity,float64,0.0,0.0,106.0,"6.8, 6.6, 6.4, 7.0, 6.9, 7.2, 6.7, 7.1, 6.5, 7.4"
2,volatile_acidity,float64,0.0,0.0,187.0,"0.28, 0.24, 0.26, 0.25, 0.22, 0.27, 0.23, 0.2, 0.3, 0.32"
3,citric_acid,float64,0.0,0.0,89.0,"0.3, 0.28, 0.32, 0.49, 0.26, 0.34, 0.29, 0.27, 0.24, 0.31"
4,residual_sugar,float64,0.0,0.0,316.0,"2.0, 1.8, 1.6, 1.4, 1.2, 2.2, 2.1, 1.9, 1.7, 1.5"
5,chlorides,float64,0.0,0.0,214.0,"0.044, 0.036, 0.042, 0.046, 0.04, 0.05, 0.048, 0.047, 0.045, 0.038"
6,free_sulfur_dioxide,float64,0.0,0.0,135.0,"29.0, 6.0, 26.0, 15.0, 31.0, 24.0, 17.0, 34.0, 35.0, 23.0"
7,total_sulfur_dioxide,float64,0.0,0.0,276.0,"111.0, 113.0, 122.0, 117.0, 124.0, 98.0, 114.0, 128.0, 118.0, 150.0"
8,density,float64,0.0,0.0,998.0,"0.9972, 0.9976, 0.992, 0.998, 0.9928, 0.9986, 0.9962, 0.9966, 0.9968, 0.9956"
9,pH,float64,0.0,0.0,108.0,"3.16, 3.14, 3.22, 3.2, 3.15, 3.19, 3.18, 3.24, 3.1, 3.12"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
fixed_acidity,6497.0,7.215307,1.296434,3.80000,15.90000
volatile_acidity,6497.0,0.339666,0.164636,0.08000,1.58000
citric_acid,6497.0,0.318633,0.145318,0.00000,1.66000
residual_sugar,6497.0,5.443235,4.757804,0.60000,65.80000
chlorides,6497.0,0.056034,0.035034,0.00900,0.61100
free_sulfur_dioxide,6497.0,30.525319,17.749400,1.00000,289.00000
total_sulfur_dioxide,6497.0,115.744574,56.521855,6.00000,440.00000
density,6497.0,0.994697,0.002999,0.98711,1.03898
pH,6497.0,3.218501,0.160787,2.72000,4.01000
sulphates,6497.0,0.531268,0.148806,0.22000,2.00000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                     
wine_color 1     white   4898  75.39
           2       red   1599  24.61

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.19,-0.377,0.763,0.023,log,40291.2,2306731.9,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to wine_quality/019d736a-03ee-7297-95da-6bfa50c97f79
019d736a-03ee-7297-95da-6bfa50c97f79
1c0c6f63523ee66517fc097f5aa504f60c5d71314b22ed1561deab4f1ce6c424
